>>Jahnvi Gangwar <br>
>>102003372 <br>
>>CO!5

Get Dataset from here https://www.muratkoklu.com/datasets/

Check Feature Sparsity/Density

Check for Highly Correlated Features

Visualising Dataset.

Apply DT using SK Learn.

Apply DTusing cuml.

Apply K-Means using SK Learn

Apply K-Means Using cuml

Apply PCA using SKlearn

Apply PCA using cuml

Check for Performance Comparison

Submit the Code file here. https://forms.gle/EyugP9qVQzSAYSAV9

Best of Luck.

###libraries 

In [ ]:
# import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns

###for GPU

In [ ]:
!pip install cudf-cu11 dask-cudf-cu11 --extra-index-url=https://pypi.nvidia.com

In [ ]:
train_size = 0.985

In [ ]:
!pip install cuml-cu11 --extra-index-url=https://pypi.nvidia.com

In [ ]:
!pip install cugraph-cu11 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import cudf
import cuml
import os

#1. dataset: Date_Fruit_Datasets.csv

In [ ]:
df = pd.read_csv("/content/Date_Fruit_Datasets.csv")

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
# df = df.drop(['Class'], axis=1)
from sklearn.preprocessing import LabelEncoder

# create a LabelEncoder object
le = LabelEncoder()
df['Class'] = le.fit_transform(df['Class'])



In [ ]:
df.head()

# 2. Check Feature Sparsity/Density

In [ ]:
# Calculate the density of the DataFrame
density = df.astype(bool).sum(axis=1).sum() / df.size

# Print the density and sparsity of the DataFrame
print("Density of DataFrame: {:.2f}%".format(density * 100))
print("Sparsity of DataFrame: {:.2f}%".format((1 - density) * 100))


In [ ]:
# Create a heatmap of the DataFrame 
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(df, cmap='binary', aspect='auto')
ax.set_title("Feature Sparsity/Density\nDensity: {:.2f}%".format(density * 100))
plt.xticks(range(df.shape[1]), df.columns)
plt.yticks(range(df.shape[0]), df.index)
plt.show()

#3.Check for Highly Correlated Features

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix
corr_matrix = df.corr()

# Set the correlation threshold
corr_threshold = 0.8

# Find the highly correlated features
corr_pairs = [(i, j) for i in range(corr_matrix.shape[0]) for j in range(i+1, corr_matrix.shape[1]) if abs(corr_matrix.iloc[i,j]) >= corr_threshold]

# Print the highly correlated features
if corr_pairs:
    for pair in corr_pairs:
        print("Features {} and {} are highly correlated (correlation coefficient: {:.2f})".format(df.columns[pair[0]], df.columns[pair[1]], corr_matrix.iloc[pair[0], pair[1]]))
else:
    print("No highly correlated features found.")


In [ ]:
# Plot the correlation matrix as a heatmap
sns.set(style="white")
fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, ax=ax)

plt.show()


#4.Visualising Dataset

In [ ]:
# Plot all columns in df
df.plot()
plt.show()


#5, 6.Apply DT using SK learn and cuml and then Random Forest for both with Comparision [Question 11]

## Decision Tree sklearn

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=13)

In [ ]:
dt = DecisionTreeClassifier()

In [ ]:
dt.fit(X_train, y_train)

In [ ]:
y_pred = dt.predict(X_test)


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")


##Random Forest

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from cuml.ensemble import RandomForestClassifier as cuRF

In [ ]:
# Split the data into features (X) and labels (y)
X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
# Split the data into training and testing sets
train_rows = int(train_size * df.shape[0])
X_train, X_test = X[:train_rows], X[train_rows:]
y_train, y_test = y[:train_rows], y[train_rows:]

In [ ]:
# Build a scikit-learn random forest classifier
rf_sk = RandomForestClassifier(n_estimators=100)
rf_sk.fit(X_train, y_train)
acc_sk = rf_sk.score(X_test, y_test)

In [ ]:
# Build a cuML random forest classifier
rf_cu = cuRF(n_estimators=100)
rf_cu.fit(X_train, y_train)
acc_cu = rf_cu.score(X_test, y_test)

In [ ]:
print(f"Accuracy (scikit-learn): {acc_sk:.4f}")
print(f"Accuracy (cuML): {acc_cu:.4f}")

In [ ]:
# Plot the accuracy comparison
labels = ['scikit-learn', 'cuML']
accuracy = [acc_sk, acc_cu]
x = np.arange(len(labels))
width = 0.35

In [ ]:
fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, accuracy, width, label='Accuracy')

ax.set_ylabel('Accuracy')
ax.set_title('Accuracy comparison between scikit-learn and cuML')
ax.set_xticks(x)
ax.legend()

fig.tight_layout()
plt.show()

#9, 10. Apply PCA using SKlearn and cuml and Comparisoin [Question11]

In [ ]:
from sklearn.decomposition import PCA
from cuml import PCA as cuPCA

In [ ]:
# Split the data into features (X) and labels (y)
X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
# Build a scikit-learn PCA model
pca_sk = PCA(n_components=3)
X_sk = pca_sk.fit_transform(X)
var_sk = pca_sk.explained_variance_ratio_

In [ ]:
# Build a cuML PCA model
pca_cu = cuPCA(n_components=3)
X_cu = pca_cu.fit_transform(X)
var_cu = pca_cu.explained_variance_ratio_

In [ ]:
print(f"Explained Variance Ratio (scikit-learn): {var_sk}")
print(f"Explained Variance Ratio (cuML): {var_cu}")


In [ ]:
X_sk

In [ ]:
X_cu

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt

# Generate scatter plot for scikit-learn
plt.figure(figsize=(8, 6))
plt.scatter(X_sk[:, 0], X_sk[:, 1], c=y, cmap='plasma')
plt.title('Scikit-learn PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

In [ ]:
from cuml.decomposition import PCA

# Convert the Pandas DataFrame to a cuDF dataframe
df_cu = cudf.DataFrame.from_pandas(df)

# Create a cuML PCA object
pca = PCA(n_components=2)

# Fit the PCA model to the data
pca.fit(df_cu)

# Transform the data using the PCA model
df_pca = pca.transform(df_cu)

# Convert the transformed data to a NumPy array
df_pca_np = df_pca.to_numpy()

# Generate scatter plot for cuML
plt.figure(figsize=(8, 6))
plt.scatter(df_pca_np[:, 0], df_pca_np[:, 1], cmap='plasma')
plt.title('cuML PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()


#8.Apply K-Means Using cuml

In [ ]:
from time import time

In [ ]:
df = pd.read_csv("/content/Date_Fruit_Datasets.csv")
# df = df.drop(['Class'], axis=1)
from sklearn.preprocessing import LabelEncoder

# create a LabelEncoder object
le = LabelEncoder()
df['Class'] = le.fit_transform(df['Class'])



In [ ]:


from cuml.cluster import KMeans as KMeans

kmeans = KMeans(n_clusters=10, random_state=0)
cufit_lines=[]
cufit_time=[]
for i in range(0,len(df)):
  tic = time()
  clusters = kmeans.fit(df)
  toc = time()
  running_time = (toc - tic) * 1000
  cufit_lines.append(len(df))
  cufit_time.append(running_time)

#7.Apply K-Means using SK Learn


In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters = 10, random_state = 0, n_init= 'auto')
skfit_lines=[]
skfit_time=[]
tic = time()
clusters = kmeans.fit(df)
toc = time()
running_time = (toc - tic) * 1000
skfit_lines.append(len(df))
skfit_time.append(running_time)

In [ ]:
skfit_lines

In [ ]:
skfit_time

#11

In [ ]:
# plot lines
plt.plot(skfit_lines, skfit_time,'bo', label = "skLearn",linestyle="-.")
plt.plot(cufit_lines, cufit_time,'ro', label = "cuDF",linestyle="-.")
plt.legend()
plt.title("Run time for Kmeans using skLearn and cuDF")
plt.show()